In [ ]:
# ==============================
# Tanzila Islam
# Email: tanzilamohita@gmail.com
# ===============================

#### Import Libraries

In [ ]:
import numpy as np
import pandas as pd
from deepcgp.Data_Processing import one_hot_encode_snp_array
from deepcgp.DeepCGP import compress_data

import warnings
warnings.filterwarnings("ignore")

#### Example use of Data Compression with Demo Data

In [ ]:
# read data
data = pd.read_csv(f'Data/X.csv', index_col=0)
print(data.iloc[:5, :5])

In [ ]:
# convert to one hot encoding
snp_array = data.astype(str).values
one_hot = one_hot_encode_snp_array(snp_array)
print(one_hot)
print(one_hot.shape)

In [ ]:
best_config = {
    "InputDim": 28,
    "Compress": [14, 7, 3],
    "BatchSize": 52,
    "LearningRate": 0.001,
    "Epochs": 200,
}

X = one_hot.astype(np.float32)
compressed = compress_data(X, best_config, seed=42, verbose=True)

#### Example use of ConvCGP Model on fixed training-validation split

In [ ]:
C2 = pd.read_csv(f'Data/X_C2.csv', index_col=0)
print(C2.head())
Y = pd.read_csv(f'Data/Y.csv', index_col=0)
print(Y.head())

In [ ]:
import numpy as np
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from deepcgp.ConvCGP import create_model
from sklearn.model_selection import train_test_split
warnings.filterwarnings("ignore")

# Drop missing values
Y = Y.dropna()
X = C2.loc[Y.index]

# Preprocess X
X.columns = np.arange(0, len(X.columns))
Y.columns = np.arange(0, len(Y.columns))

# Define callbacks
es = EarlyStopping(monitor='val_loss', mode='min', patience=50, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=20, min_lr=1e-5)

corr_df = []

# Training and evaluating on X
for i in range(0, 1):

    X_train, X_valid, y_train, y_valid = train_test_split(
        X, Y[i], test_size=0.2, random_state=42
    )

    X_train = np.expand_dims(X_train, axis=2)
    X_valid = np.expand_dims(X_valid, axis=2)

    model_X = create_model(input_shape=(X_train.shape[1], 1))

    model_X.fit(
        X_train, y_train,
        epochs=400,
        batch_size=64,
        validation_data=(X_valid, y_valid),
        callbacks=[es, reduce_lr],
        shuffle=False,
        verbose=0
    )

    # Prediction
    y_hat_X = model_X.predict(X_valid, verbose=0)

    corr_X = np.corrcoef(y_valid, y_hat_X[:, 0])[0, 1]

    print(f'Correlation for trait {i} = {corr_X:.4f}')

    corr_df.append([i, corr_X])